## Build Results Fact
- 1.Read silver results table
- 2.Read silver sprints table
- 3.Add column session_type with values RACE or SPRINT
- 4.Union results and sprints
- 5.Derive addtitional columns
-  1.is_win
-  2.is_position
-  3.has_points- indicates driver has scored pints 
- 6.Write transformed data to gold table

In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table=f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
import pyspark.sql.functions as F

**Step 1 : Read sourcetables**
-  silver.results
-  silver.sprints

In [0]:
results_df=(
    spark.table(f"{catalog_name}.{silver_schema}.results")
    .withColumn("session_type",F.lit("RACE"))
    .drop("race_name","race_date","ingestion_timestamp","source_file")
)

In [0]:
sprints_df=(
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
    .withColumn("session_type",F.lit("SPRINT"))
    .drop("race_name","race_date","ingestion_timestamp","source_file")
)

**Step 2 Union results amd sprints**

In [0]:
#here neither unionall nor union removes duplicates we need to use distinct method
#unionbyname join the dataframe by column name and not position
results_sprint_df=results_df.unionByName(sprints_df)

**Step 3 Add dervied column**

In [0]:
fact_session_results_df=(
    results_sprint_df
    .withColumn("is_win",F.col("final_position")==1)
   .withColumn("is_podium",F.col("final_position").between(1,3))
   .withColumn("has_points",F.col("points")>0)
)

In [0]:
display(fact_session_results_df.filter("season=2025"))

round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points
1,2025,williams,albon,6,57,23,10.0,5,5,Finished,RACE,false,false,true
1,2025,aston_martin,alonso,12,32,14,0.0,17,R,Retired,RACE,false,false,false
1,2025,mercedes,antonelli,16,57,12,12.0,4,4,Finished,RACE,false,false,true
1,2025,haas,bearman,20,57,87,0.0,14,14,Finished,RACE,false,false,false
1,2025,sauber,bortoleto,15,45,5,0.0,16,R,Retired,RACE,false,false,false
1,2025,alpine,doohan,14,0,7,0.0,19,R,Retired,RACE,false,false,false
1,2025,alpine,gasly,9,57,10,0.0,11,11,Finished,RACE,false,false,false
1,2025,rb,hadjar,11,0,6,0.0,20,R,Retired,RACE,false,false,false
1,2025,ferrari,hamilton,8,57,44,1.0,10,10,Finished,RACE,false,false,true
1,2025,sauber,hulkenberg,17,57,27,6.0,7,7,Finished,RACE,false,false,true


**Step 4 Write transformed data to the gold fact_session_results table**

In [0]:
(
    fact_session_results_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))

round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points
1,1950,maserati,bira,5,49,21,0.0,14,R,Out of fuel,RACE,false,false,false
1,1950,lago,cabantous,6,68,14,3.0,4,4,+2 Laps,RACE,false,false,true
1,1950,maserati,chiron,11,26,19,0.0,18,R,Clutch,RACE,false,false,false
1,1950,lago,claes,21,64,18,0.0,11,11,+6 Laps,RACE,false,false,false
1,1950,alta,crossley,17,43,24,0.0,16,R,Transmission,RACE,false,false,false
1,1950,lago,etancelin,14,65,16,0.0,8,8,+5 Laps,RACE,false,false,false
1,1950,alfa,fagioli,2,70,3,6.0,2,2,Finished,RACE,false,true,true
1,1950,alfa,fangio,3,62,1,0.0,12,R,Oil leak,RACE,false,false,false
1,1950,alfa,farina,1,70,2,9.0,1,1,Finished,RACE,true,true,true
1,1950,maserati,fry,20,64,10,0.0,10,10,+6 Laps,RACE,false,false,false
